In [61]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim import corpora
from gensim.models import LdaModel
from nltk.tokenize import word_tokenize
import os
import string
import pandas as pd
import re
import pyLDAvis
import pyLDAvis.gensim_models as gensim_models

# Download necessary NLTK data if not already present
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('tokenizers/punkt_tab') # Add this line to download punkt_tab
except LookupError:
    nltk.download('punkt_tab')

In [62]:
input_dir = '/Users/freddie/TextAsData/Project/ForAnalysis'
output_dir = '/Users/freddie/TextAsData/Project/ForAnalysis/TM_Analysis/'

In [63]:
txt_files = [f for f in os.listdir(input_dir) if f.endswith('.txt')]
processed_files_count = 0
output_files = []

print(f"Found {len(txt_files)} .txt files in {input_dir}")

Found 10 .txt files in /Users/freddie/TextAsData/Project/ForAnalysis


In [64]:
# Initialize stopwords and punctuation translator
stop_words = set(stopwords.words('english'))
# Add common curly quotes, em/en dashes, and bullet point to the punctuation set
all_punctuation = string.punctuation + "‘’“”—–•"
translator = str.maketrans('', '', all_punctuation)

def remove_stopwords(text):
    """Removes English stopwords, punctuation (including curly quotes), numbers, and single-letter words from a given text."""
    # Remove punctuation (including curly quotes, dashes, and bullet point)
    text_no_punctuation = text.translate(translator)
    # Remove numbers
    text_no_numbers = re.sub(r'\d+', '', text_no_punctuation)
    word_tokens = word_tokenize(text_no_numbers)
    # Filter out stopwords and single-letter words
    filtered_words = [w for w in word_tokens if not w.lower() in stop_words and len(w) > 1]
    return " ".join(filtered_words)

# Reset processed_files_count and output_files for this run
processed_files_count = 0
output_files = []

print(f"Starting stopword, punctuation, number, and single-letter word removal for {len(txt_files)} files...")

for filename in txt_files:
    input_filepath = os.path.join(input_dir, filename)
    output_filename = f"TM-processed_{filename}"
    output_filepath = os.path.join(output_dir, output_filename)

    try:
        with open(input_filepath, 'r', encoding='utf-8') as f_in:
            content = f_in.read()

        processed_content = remove_stopwords(content)

        with open(output_filepath, 'w', encoding='utf-8') as f_out:
            f_out.write(processed_content)

        output_files.append(output_filepath)
        processed_files_count += 1
    except Exception as e:
        print(f"Error processing file {filename}: {e}")

print(f"Finished processing. Successfully processed {processed_files_count} files.")
print(f"Cleaned files are saved in: {output_dir}")


Starting stopword, punctuation, number, and single-letter word removal for 10 files...
Finished processing. Successfully processed 10 files.
Cleaned files are saved in: /Users/freddie/TextAsData/Project/ForAnalysis/TM_Analysis/


In [65]:
# Assuming 'output_files' contains the paths to the processed text files
# and 'processed_dir' is the directory where they are saved.

processed_texts = []
for filepath in output_files:
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
        processed_texts.append(word_tokenize(text))

# Create a dictionary from the processed texts
dictionary = corpora.Dictionary(processed_texts)

# Filter out words that appear in less than 3 documents (absolute number) or more than 50% of the documents (fraction)
dictionary.filter_extremes(no_below=3, no_above=0.5)

# Create a Bag-of-Words (BoW) corpus
corpus = [dictionary.doc2bow(text) for text in processed_texts]

print(f"Number of unique tokens: {len(dictionary)}")
print(f"Number of documents in corpus: {len(corpus)}")


Number of unique tokens: 6085
Number of documents in corpus: 10


In [82]:
# 7 Topics seems to be the best outcome
num_topics = 7
lda_model = LdaModel(corpus, num_topics=num_topics, id2word=dictionary, passes=15, random_state=100)

# Print topics
print("LDA Model Topics:")
for idx, topic in lda_model.print_topics(num_words=10):
    print(f"Topic #{idx + 1}: {topic}")


LDA Model Topics:
Topic #1: 0.026*"machine" + 0.024*"Royal" + 0.022*"Flying" + 0.022*"machines" + 0.017*"German" + 0.015*"Lieutenant" + 0.014*"Mr" + 0.011*"pilots" + 0.010*"Naval" + 0.010*"squadrons"
Topic #2: 0.013*"patrol" + 0.013*"etc" + 0.013*"rifle" + 0.012*"platoon" + 0.010*"target" + 0.009*"noncommissioned" + 0.008*"map" + 0.008*"squads" + 0.008*"Corporal" + 0.007*"MARCH"
Topic #3: 0.020*"Colonel" + 0.014*"Col" + 0.011*"Fort" + 0.008*"Major" + 0.008*"Gen" + 0.007*"Street" + 0.007*"Stirling" + 0.007*"Congress" + 0.007*"Sullivan" + 0.006*"Mr"
Topic #4: 0.022*"Indians" + 0.014*"Sun" + 0.012*"monarch" + 0.008*"Spain" + 0.008*"Tu" + 0.008*"emperor" + 0.006*"temple" + 0.006*"despatched" + 0.005*"ambassadors" + 0.005*"harbour"
Topic #5: 0.034*"nd" + 0.033*"Waterloo" + 0.029*"Dec" + 0.027*"Capt" + 0.026*"rd" + 0.025*"Wm" + 0.023*"Sept" + 0.023*"Jan" + 0.022*"Aug" + 0.020*"Oct"
Topic #6: 0.040*"Catherine" + 0.014*"doctor" + 0.013*"Thats" + 0.011*"cant" + 0.011*"priest" + 0.011*"Miss" + 0

In [83]:
# Prepare list to hold data for all topics
all_topic_data = []

# Regex to capture score and word from the topic string
pattern = re.compile(r'(\d+\.\d+)\*"([^"]+)"')

print("Generating topic table...")

# Iterate through each topic returned by LDA model
for idx, topic_str in lda_model.print_topics(num_words=10):
    topic_words = []
    topic_scores = []

    # Parse the topic string to extract words and scores
    for match in pattern.finditer(topic_str):
        score = float(match.group(1))
        word = match.group(2)
        topic_words.append(word)
        topic_scores.append(score)

    # Create a temporary DataFrame for the current topic's words and scores
    topic_df = pd.DataFrame({
        f'Topic {idx + 1} Words': topic_words,
        f'Topic {idx + 1} Score': topic_scores
    })
    all_topic_data.append(topic_df)

# Concatenate all individual topic DataFrames horizontally
final_topics_df = pd.concat(all_topic_data, axis=1)

print("LDA Topics Table:")
display(final_topics_df)


Generating topic table...
LDA Topics Table:


,Topic 1 Words,Topic 1 Score,Topic 2 Words,Topic 2 Score,Topic 3 Words,Topic 3 Score,Topic 4 Words,Topic 4 Score,Topic 5 Words,Topic 5 Score,Topic 6 Words,Topic 6 Score,Topic 7 Words,Topic 7 Score
0,machine,0.026,patrol,0.013,Colonel,0.020,Indians,0.022,nd,0.034,Catherine,0.040,Sherman,0.017
1,Royal,0.024,etc,0.013,Col,0.014,Sun,0.014,Waterloo,0.033,doctor,0.014,Sheridan,0.010
2,Flying,0.022,rifle,0.013,Fort,0.011,monarch,0.012,Dec,0.029,Thats,0.013,Lee,0.009
3,machines,0.022,platoon,0.012,Major,0.008,Spain,0.008,Capt,0.027,cant,0.011,railroad,0.009
4,German,0.017,target,0.010,Gen,0.008,Tu,0.008,rd,0.026,priest,0.011,Richmond,0.008
5,Lieutenant,0.015,noncommissioned,0.009,Street,0.007,emperor,0.008,Wm,0.025,Miss,0.011,Fort,0.008
6,Mr,0.014,map,0.008,Stirling,0.007,temple,0.006,Sept,0.023,lovely,0.010,James,0.006
7,pilots,0.011,squads,0.008,Congress,0.007,despatched,0.006,Jan,0.023,girl,0.009,Smith,0.006
8,Naval,0.010,Corporal,0.008,Sullivan,0.007,ambassadors,0.005,Aug,0.022,car,0.009,Potomac,0.006
9,squadrons,0.010,MARCH,0.007,Mr,0.006,harbour,0.005,Oct,0.020,boy,0.009,Mr,0.005


In [84]:
## Prepare the visualization
vis = gensim_models.prepare(lda_model, corpus, dictionary, R=25)

# Display the visualization
pyLDAvis.display(vis)
